# 📱 Pocket OTC AI Analyzer — APK Builder

يبني APK مباشرة من GitHub داخل Google Colab بدون Telegram وبدون GitHub Actions.

READ-ONLY: التطبيق لا يسجل الدخول إلى Pocket Option ولا ينفذ صفقات.

In [ ]:
# 🚀 بناء APK بنقرة واحدة
import os, shutil, subprocess, time, traceback, zipfile, stat

REPO='https://github.com/mohmb142/Jjjjjjj.git'
ZIP_URL='https://github.com/mohmb142/Jjjjjjj/archive/refs/heads/main.zip'
ROOT='/content/Jjjjjjj'
ANDROID=os.path.join(ROOT,'phone-agent')

def run(cmd, **kwargs):
    print('▶️', ' '.join(map(str,cmd)))
    return subprocess.run(cmd, check=True, **kwargs)

try:
    if os.path.exists(ROOT): shutil.rmtree(ROOT, ignore_errors=True)
    print('1/5 ⬇️ تنزيل المشروع...')
    try:
        run(['git','clone','--depth','1',REPO,ROOT], timeout=180)
    except Exception:
        print('⚠️ git clone فشل، استخدام ZIP...')
        zp='/content/Jjjjjjj.zip'
        run(['wget','-q','-O',zp,ZIP_URL], timeout=180)
        with zipfile.ZipFile(zp) as z: z.extractall('/content')
        os.rename('/content/Jjjjjjj-main',ROOT)
    if not os.path.isfile(os.path.join(ANDROID,'app','build.gradle.kts')):
        raise RuntimeError('مشروع Android غير موجود في phone-agent/')

    print('2/5 ☕ تجهيز Java...')
    run(['apt-get','update','-qq'], timeout=240)
    run(['apt-get','install','-y','-qq','openjdk-17-jdk','wget','unzip'], timeout=240)
    os.environ['JAVA_HOME']='/usr/lib/jvm/java-17-openjdk-amd64'
    os.environ['PATH']=os.environ['JAVA_HOME']+'/bin:'+os.environ['PATH']
    run(['java','-version'])

    print('3/5 📦 تجهيز Gradle...')
    gradle_bin='/content/gradle-8.7/bin/gradle'
    if not os.path.exists(gradle_bin):
        gz='/content/gradle.zip'
        run(['wget','-q','https://services.gradle.org/distributions/gradle-8.7-bin.zip','-O',gz], timeout=240)
        with zipfile.ZipFile(gz) as z: z.extractall('/content')
    os.chmod(gradle_bin, os.stat(gradle_bin).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    print('Gradle:', subprocess.check_output([gradle_bin,'--version'], text=True).splitlines()[0:5])

    print('4/5 🔨 بناء APK...')
    run([gradle_bin,'assembleDebug','--no-daemon','--stacktrace'], cwd=ANDROID, timeout=900)
    apk=os.path.join(ANDROID,'app','build','outputs','apk','debug','app-debug.apk')
    if not os.path.isfile(apk) or os.path.getsize(apk)==0: raise RuntimeError('لم يتم إنشاء APK')
    print(f'حجم APK: {os.path.getsize(apk)/1024/1024:.2f} MB')

    print('5/5 📱 تحميل APK...')
    from google.colab import files
    files.download(apk)
    print('✅ تم إنشاء APK بنجاح — بدون Telegram — Android 8+ / Android 10 مدعوم.')
except Exception:
    print('❌ فشل البناء:')
    traceback.print_exc()
    print('المشكلة السابقة كانت PermissionError على gradle؛ تم إصلاح صلاحية التنفيذ في هذه النسخة.')